# Who Gets Power First? A Classification Model
### Quick Start Orange County: AI Design Contest — Week 3

---

Your regression model predicts *how much* power the grid needs. But prediction alone
is not enough when supply runs short.

During an emergency — a hurricane, a heat wave, a mass casualty event — your smart city
cannot power everyone at full capacity. Someone has to wait. The question is: **who?**

That is a classification problem. You are not predicting a number. You are assigning
each power user to a **priority tier** — a category that determines their place in the
restoration queue when the grid is stressed.

**Why classification?** Because the question is *which group does this belong to?*
not *how much?* The model learns to sort power users based on their consumption
patterns — the same patterns you drew in Week 1.

**The stakes are real.** A hospital miscategorized as Non-Critical during an outage
is a life-safety failure. A school miscategorized as Critical wastes scarce power
that a fire station needed. Every classification error has a consequence.

**By the end of this notebook your team will have:**
- Defined and justified your own priority tier labels
- Engineered features that capture what makes a building critical
- Trained and evaluated a decision tree classifier
- Interpreted the confusion matrix as a real-world risk assessment
- Written a reflection connecting misclassification to human consequences

**Keep your Quick Reference sheet open — the power restoration priority order
is the framework for your labeling decisions.**

---

**[INSTRUCTOR NOTE: Insert Classification companion reading here]**

---

## Setup

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    ConfusionMatrixDisplay, accuracy_score
)

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['font.size'] = 11

# Color palette — one color per priority tier, consistent throughout
TIER_COLORS = {
    'Critical':     '#E53935',   # red
    'Flexible':     '#FB8C00',   # orange  
    'Non_Critical': '#43A047',   # green
}

print(f"Looking for data in: {os.getcwd()}")
print("Libraries loaded successfully")

In [ ]:
# Load the classification dataset
df = pd.read_csv('smart_city_classification.csv')

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nBuilding types in dataset:")
print(sorted(df['building_type'].unique()))
print(f"\nFirst 5 rows:")
df.head()

---
## Part 1: Understanding the Dataset

This dataset is structured differently from the regression dataset.
Each row is one building type at one time interval — 19 building types
× 200 time steps = 3,800 rows.

The features describe each building's energy consumption behavior.
The target is `priority_tier` — the label your model will learn to predict.

Before anything else — inspect the data.

In [ ]:
# Dataset structure
print("Data types:")
print(df.dtypes)
print(f"\nMissing values: {df.isnull().sum().sum()}")
print(f"\nLabel distribution:")
print(df['priority_tier'].value_counts())
print(f"\nBuildings per tier:")
print(df.groupby('priority_tier')['building_type'].nunique())

In [ ]:
# Visualize energy demand by priority tier
# This shows what the model will use to distinguish tiers

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, (tier, color) in enumerate(TIER_COLORS.items()):
    tier_data = df[df['priority_tier'] == tier]
    axes[i].hist(tier_data['energy_kWh'], bins=40, 
                 color=color, alpha=0.75, edgecolor='white')
    axes[i].set_title(f'{tier}\n({tier_data["building_type"].nunique()} building types)')
    axes[i].set_xlabel('Energy (kWh per 15-min interval)')
    axes[i].set_ylabel('Count')
    axes[i].axvline(tier_data['energy_kWh'].mean(), 
                    color='black', linestyle='--', linewidth=2,
                    label=f'Mean: {tier_data["energy_kWh"].mean():.1f}')
    axes[i].legend(fontsize=9)

plt.suptitle('Energy Demand Distribution by Priority Tier', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Per-building summary — what does each building type look like?
building_summary = df.groupby(['building_type', 'priority_tier']).agg(
    mean_kWh=('energy_kWh', 'mean'),
    max_kWh=('energy_kWh', 'max'),
    std_kWh=('energy_kWh', 'std'),
    variability=('energy_kWh', lambda x: x.std() / x.mean())
).round(2).reset_index()

# Sort by mean demand descending
building_summary = building_summary.sort_values('mean_kWh', ascending=False)
print("Building types by energy demand:")
print(building_summary[['building_type','priority_tier','mean_kWh','std_kWh','variability']].to_string(index=False))

In [ ]:
# Exercise 1.1 — Reading the data before modeling

# Q1: Look at the building_summary table above.
#     Does mean energy demand alone separate the three tiers cleanly?
#     Give a specific example where two buildings from different tiers
#     have similar mean demand.
# A: 

# Q2: Look at the variability column (std/mean).
#     Which tier tends to have the LOWEST variability? 
#     What does low variability tell you about how that type of building operates?
# A: 

# Q3: Police station and fire station are in the Critical tier
#     but have relatively LOW mean energy demand.
#     If energy demand alone doesn't make them critical, what does?
#     This is a domain knowledge question — not a data question.
# A: 

# Q4: Look at the histogram plots. Which tier has the widest spread of values?
#     What does that tell you about how hard it will be to classify that tier?
# A: 

# Q5: Before training any model — based only on what you can see in the data —
#     which buildings do you think the model will struggle to classify correctly?
#     Write your prediction here. You will check it after training.
# A: 

---
## Part 2: Your Priority Tier Labels

The dataset comes with pre-assigned priority tiers. But your team defined your own
priority tiers in Task 5 of Week 1 — before you saw any model results.

Now you need to reconcile those two things. Do you agree with the pre-assigned labels?
If not, this is the moment to change them — with justification.

**This is a genuine design decision.** There is no single correct answer.
Different teams will make different choices, and judges will ask you to defend yours.

Use your Quick Reference sheet — the power restoration priority order is the
framework. But remember: that order was designed for a traditional grid.
Your smart city might make different choices.

In [ ]:
# Current label assignments — review and modify if your team disagrees
# Format: 'building_type': 'tier'
# Tiers must be exactly: 'Critical', 'Flexible', or 'Non_Critical'

# REVIEW EACH ASSIGNMENT AND DECIDE IF YOU AGREE
# Write your reasoning for any changes in the comments below

priority_labels = {
    # Critical — power restored first, life-safety or essential infrastructure
    'hospital':         'Critical',    # agree / disagree? reasoning:
    'clinic':           'Critical',    # agree / disagree? reasoning:
    'police_station':   'Critical',    # agree / disagree? reasoning:
    'fire_station':     'Critical',    # agree / disagree? reasoning:
    
    # Flexible — power restored second, important but can tolerate delay
    'college':          'Flexible',    # agree / disagree? reasoning:
    'k12_school':       'Flexible',    # agree / disagree? reasoning:
    'commercial_office':'Flexible',    # agree / disagree? reasoning:
    'library':          'Flexible',    # agree / disagree? reasoning:
    'warehouse':        'Flexible',    # agree / disagree? reasoning:
    
    # Non_Critical — power restored last
    'restaurant':       'Non_Critical',# agree / disagree? reasoning:
    'hotel':            'Non_Critical',# agree / disagree? reasoning:
    'grocery':          'Non_Critical',# agree / disagree? reasoning: 
    'fast_food':        'Non_Critical',# agree / disagree? reasoning:
    'post_office':      'Non_Critical',# agree / disagree? reasoning:
    'shopping_mall':    'Non_Critical',# agree / disagree? reasoning:
    'fitness_center':   'Non_Critical',# agree / disagree? reasoning:
    'bus_station':      'Non_Critical',# agree / disagree? reasoning:
    'movie_theater':    'Non_Critical',# agree / disagree? reasoning:
    'gas_station':      'Non_Critical',# agree / disagree? reasoning:
}

# Apply your labels to the dataset
df['priority_tier'] = df['building_type'].map(priority_labels)

print("Your priority tier assignments:")
print("-" * 45)
for tier in ['Critical', 'Flexible', 'Non_Critical']:
    buildings = [b for b, t in priority_labels.items() if t == tier]
    print(f"\n{tier} ({len(buildings)} buildings):")
    for b in buildings:
        print(f"  • {b}")

In [ ]:
# Exercise 2.1 — Justify your most difficult labeling decisions

# Q1: Which building type was hardest to assign to a tier? Why?
#     Consider: grocery stores during a hurricane provide food for
#     emergency responders. Does that change their tier?
# A: 

# Q2: A school becomes an emergency shelter during a hurricane.
#     Should that change its priority tier? How would your smart city
#     system handle a building whose priority changes depending on context?
# A: 

# Q3: Your team assigned ___ buildings to Critical tier.
#     During a severe outage, the grid can only support 30% of normal capacity.
#     If your Critical tier consumes more than 30% of grid capacity,
#     what happens? Was your Critical tier too large?
# A: 

# Q4: Who should make these labeling decisions in a real smart city system?
#     The AI engineer? The utility company? The city government? Citizens?
#     What are the implications of each?
# A: 

---
## Part 3: Feature Engineering for Classification

The features that matter for classification are different from regression.
For regression, you wanted features that predicted *how much* demand would be.
For classification, you want features that reveal *what kind* of building this is.

Think about what distinguishes a Critical building from a Non-Critical one:
- Critical buildings run at high demand **around the clock** — low variability
- Non-Critical buildings have **peaks and valleys** — high variability
- Flexible buildings are somewhere in between

Your features need to capture that behavioral signature.

In [ ]:
# Feature 1: variability_ratio (already in dataset as global_std_kWh / global_mean_kWh)
# Hospitals run 24/7 — low variability
# Restaurants peak at meal times — high variability
# This ratio captures operational pattern better than raw demand

df['variability_ratio'] = df['global_std_kWh'] / (df['global_mean_kWh'] + 1e-6)

# Verify — Critical buildings should have LOWER variability
print("Average variability ratio by tier:")
print(df.groupby('priority_tier')['variability_ratio'].mean().round(3))
print("\nDoes lower variability = more Critical? (should be yes)")

In [ ]:
# Feature 2: overnight_demand_ratio
# Critical buildings (hospitals) maintain HIGH demand overnight
# Non-critical buildings (restaurants, shops) go quiet at night
# This feature captures whether a building is truly 24/7 or business-hours only

# YOUR CODE: Create a column that is 1 if hour is between 22 and 6 (overnight), else 0
# Hint: use (df['hour'] >= 22) | (df['hour'] <= 6)
# then compare energy_kWh to global_mean_kWh during overnight hours

df['is_overnight'] = REPLACE_THIS_WITH_YOUR_CONDITION

# Given: compute overnight demand ratio per building
overnight_mean = df[df['is_overnight'] == 1].groupby('building_type')['energy_kWh'].mean()
df['overnight_mean_kWh'] = df['building_type'].map(overnight_mean)
df['overnight_demand_ratio'] = df['overnight_mean_kWh'] / (df['global_mean_kWh'] + 1e-6)

# Justification (one sentence — why does overnight behavior help classify priority?):
#

print("Average overnight demand ratio by tier:")
print(df.groupby('priority_tier')['overnight_demand_ratio'].mean().round(3))
print("\nHospitals should have ratio near 1.0 (demand stays high overnight)")
print("Restaurants should have ratio well below 1.0 (quiet overnight)")

In [ ]:
# Feature 3: peak_to_baseline_ratio
# How much does demand spike above its own baseline?
# Critical buildings have small spikes — they're always running hard
# Non-critical buildings have large spikes — busy periods vs quiet periods

# YOUR CODE: Create peak_to_baseline_ratio
# Formula: global_max_kWh divided by global_mean_kWh
# Example pattern: df['col'] = df['col_a'] / (df['col_b'] + 1e-6)

df['peak_to_baseline_ratio'] = REPLACE_THIS_WITH_YOUR_FORMULA

# Justification (one sentence):
#

print("Average peak-to-baseline ratio by tier:")
print(df.groupby('priority_tier')['peak_to_baseline_ratio'].mean().round(3))
print("\nLower ratio = more stable = more likely Critical")

In [ ]:
# Exercise 3.1 — Design your own classification feature
# Create ONE feature that captures something the above features miss.
#
# Ideas:
#   - business_hours_intensity: ratio of business-hours demand to overnight demand
#   - demand_rank: rank each building by mean demand (1=highest)
#   - is_24_7: binary flag for buildings that never drop below 80% of their mean
#   - weekend_drop: how much demand falls on weekends vs weekdays
#
# Example patterns:
#   df['col'] = df['col_a'] / df['col_b']           # ratio
#   df['col'] = (df['col_a'] > threshold).astype(int) # binary flag
#   df['col'] = df['col_a'].rank(pct=True)            # percentile rank

# Feature name: (rename 'my_class_feature' to something descriptive)
# YOUR CODE:
df['my_class_feature'] = REPLACE_THIS_WITH_YOUR_FORMULA

# Why did you create this feature? What behavioral pattern does it capture?
# JUSTIFICATION:
#

print("My feature by tier:")
print(df.groupby('priority_tier')['my_class_feature'].mean().round(3))
print("\nDoes this feature separate the tiers? Does the direction make sense?")

In [ ]:
# Visualize your engineered features
# Do they actually separate the tiers visually?

eng_features = ['variability_ratio', 'overnight_demand_ratio', 
                'peak_to_baseline_ratio', 'my_class_feature']

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for i, feat in enumerate(eng_features):
    for tier, color in TIER_COLORS.items():
        tier_data = df[df['priority_tier'] == tier][feat].dropna()
        axes[i].hist(tier_data, bins=30, color=color, alpha=0.6,
                     edgecolor='white', label=tier)
    axes[i].set_title(feat)
    axes[i].set_xlabel('Feature Value')
    axes[i].set_ylabel('Count')
    axes[i].legend(fontsize=8)

plt.suptitle('Engineered Features by Priority Tier\n'
             'Good features show clear separation between colors',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

---
## Part 4: Feature Selection and Model Training

A **decision tree** classifies by asking a series of yes/no questions about your features.
At each branch it asks the question that best separates the classes — like a flowchart
that the data taught itself.

Decision trees are interpretable — you can read the rules the model learned and ask
*does this make sense?* That matters enormously when the model's decisions affect
who gets power during an emergency.

In [ ]:
# Exercise 4.1 — YOUR FEATURE SELECTION
# Choose which features to include. You must include at least 4.
# Write your reasoning for each included AND excluded feature.
#
# Available features:
# 'energy_kWh', 'global_mean_kWh', 'global_max_kWh', 'global_std_kWh',
# 'hour', 'is_business_hours', 'pct_of_daily_max',
# 'variability_ratio', 'overnight_demand_ratio', 
# 'peak_to_baseline_ratio', 'my_class_feature'

# FEATURES WE ARE INCLUDING AND WHY:
# 1. global_mean_kWh     — because:
# 2. global_std_kWh      — because:
# 3. variability_ratio   — because:
# 4.                     — because:
# 5.                     — because:

# FEATURES WE ARE EXCLUDING AND WHY:
# 1.                     — because:
# 2.                     — because:

selected_features = [
    'global_mean_kWh',
    'global_std_kWh',
    'variability_ratio',
    # ADD MORE FEATURE NAMES HERE AS STRINGS
]

X = df[selected_features].fillna(0)
y = df['priority_tier']

print(f"Selected {len(selected_features)} features: {selected_features}")
print(f"X shape: {X.shape}")
print(f"\nClass distribution:")
print(y.value_counts())

In [ ]:
# Train / test split — stratified to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training rows: {len(X_train):,}")
print(f"Test rows:     {len(X_test):,}")
print(f"\nClass balance in test set:")
print(y_test.value_counts())
print(f"\nNote: stratify=y ensures each class is proportionally represented")
print(f"in both training and test sets. Without this, rare classes could")
print(f"be underrepresented in training — the model would never learn them.")

In [ ]:
# Exercise 4.2 — Choose your tree depth
# max_depth controls how many questions the tree can ask.
# Shallow trees (depth 2-3) are simple but may miss patterns.
# Deep trees (depth 8+) may memorize training data but fail on new data.
#
# Try at least TWO different depths and compare results.
# Choose the depth you will use for your final model and justify it.

# YOUR DEPTH EXPERIMENTS:
for depth in [3, 5, 8]:
    dt_temp = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt_temp.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, dt_temp.predict(X_train))
    test_acc  = accuracy_score(y_test,  dt_temp.predict(X_test))
    print(f"max_depth={depth}: train={train_acc:.3f}  test={test_acc:.3f}  "
          f"gap={train_acc-test_acc:.3f}")

print()
print("A large gap between train and test accuracy suggests overfitting.")
print("Your chosen depth and reasoning:")
# CHOSEN DEPTH:
# REASONING:

In [ ]:
# Train your final model
# Replace 5 with your chosen depth from the experiment above
CHOSEN_DEPTH = 5   # REPLACE WITH YOUR CHOSEN DEPTH

model = DecisionTreeClassifier(max_depth=CHOSEN_DEPTH, random_state=42)
model.fit(X_train, y_train)

print(f"Decision tree trained (max_depth={CHOSEN_DEPTH})")
print(f"\nWhat the model learned — feature importances:")
print("-" * 45)
for feat, imp in sorted(zip(selected_features, model.feature_importances_),
                         key=lambda x: x[1], reverse=True):
    bar = '█' * int(imp * 30)
    print(f"{feat:<28} {imp:.3f}  {bar}")

In [ ]:
# Read the decision tree rules
# This is what makes decision trees powerful for high-stakes decisions:
# you can read exactly what the model is doing and ask if it makes sense

tree_rules = export_text(model, feature_names=selected_features)
print("Decision Tree Rules:")
print("(Each branch is a yes/no question. Follow the path to get a classification.)")
print("-" * 60)
print(tree_rules)

In [ ]:
# Visualize the tree — the painter's view of the model's logic
fig, ax = plt.subplots(figsize=(18, 8))
plot_tree(
    model,
    feature_names=selected_features,
    class_names=model.classes_,
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax,
    impurity=False
)
ax.set_title(f'Decision Tree — Priority Tier Classification (max_depth={CHOSEN_DEPTH})',
             fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Exercise 4.3 — Reading the tree

# Q1: What is the FIRST question the tree asks (the root node)?
#     Does that make intuitive sense as the most important distinguishing factor?
# A: 

# Q2: Which feature has the highest importance score?
#     Does that match your hypothesis from the Drawing with Data notebook?
# A: 

# Q3: Trace the path for a hospital:
#     global_mean_kWh ≈ 492, global_std_kWh ≈ 88, variability_ratio ≈ 0.18
#     Follow the tree rules above — which tier does the model assign it?
#     Is that correct?
# A: 

# Q4: Could you explain this tree's logic to a city council member
#     who has no data science background? Try in 2-3 sentences.
# A: 

---
## Part 5: Evaluation — The Confusion Matrix

For classification, the confusion matrix is more informative than a single accuracy number.
It shows you **which classes get confused with which** — and in a power emergency,
not all errors are equal.

- Classifying a **hospital as Non-Critical** = life-safety failure
- Classifying a **movie theater as Critical** = wasted scarce power
- Classifying a **school as Critical** instead of Flexible = minor overallocation

The confusion matrix lets you see which errors your model makes —
and decide whether you can live with them.

In [ ]:
# Generate predictions
y_pred = model.predict(X_test)

# Overall accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Overall accuracy: {accuracy:.3f} ({accuracy*100:.1f}%)")
print(f"\nPer-class performance:")
print("-" * 50)
print(classification_report(y_test, y_pred, 
      target_names=['Critical', 'Flexible', 'Non_Critical']))

In [ ]:
# Plot the confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Raw counts
cm = confusion_matrix(y_test, y_pred, 
                       labels=['Critical', 'Flexible', 'Non_Critical'])
disp1 = ConfusionMatrixDisplay(confusion_matrix=cm,
                                display_labels=['Critical', 'Flexible', 'Non_Critical'])
disp1.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix — Raw Counts\n'
                  'Diagonal = correct classifications')

# Normalized (percentages)
cm_norm = confusion_matrix(y_test, y_pred,
                            labels=['Critical', 'Flexible', 'Non_Critical'],
                            normalize='true')
disp2 = ConfusionMatrixDisplay(confusion_matrix=cm_norm,
                                display_labels=['Critical', 'Flexible', 'Non_Critical'])
disp2.plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('Confusion Matrix — Normalized\n'
                  'What % of each class is correctly classified?')

plt.suptitle('Classification Results by Priority Tier', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Which specific buildings are misclassified?
# This is the most important diagnostic for a high-stakes system

results = X_test.copy()
results['building_type'] = df.loc[X_test.index, 'building_type'].values
results['actual_tier']    = y_test.values
results['predicted_tier'] = y_pred
results['correct']        = results['actual_tier'] == results['predicted_tier']

misclassified = results[~results['correct']]

print("Misclassified buildings:")
print("-" * 55)
if len(misclassified) == 0:
    print("No misclassifications! (Check your features — this may indicate overfitting)")
else:
    summary = misclassified.groupby(
        ['building_type', 'actual_tier', 'predicted_tier']
    ).size().reset_index(name='count')
    summary['error_type'] = summary.apply(
        lambda r: '⚠️ DANGEROUS' 
        if r['actual_tier'] == 'Critical' and r['predicted_tier'] != 'Critical'
        else '⚡ WASTEFUL' 
        if r['actual_tier'] != 'Critical' and r['predicted_tier'] == 'Critical'
        else 'minor', axis=1
    )
    print(summary.to_string(index=False))

In [ ]:
# Exercise 5.1 — The confusion matrix as a risk assessment

# Q1: Look at the normalized confusion matrix.
#     What percentage of Critical buildings were correctly classified?
#     Is that acceptable for a life-safety system? What would be?
# A: 

# Q2: Are any Critical buildings being misclassified as Non_Critical?
#     (Look at the top-right cell of the confusion matrix.)
#     If yes — which building, and what would happen in a real emergency?
# A: 

# Q3: The misclassified buildings table labels errors as DANGEROUS, WASTEFUL, or minor.
#     Which type of error does your model make most?
#     Which type is worse for the community?
# A: 

# Q4: Your model has overall accuracy of ___%. A city official asks:
#     'Is this model ready to deploy?' What would you tell them?
#     Use specific numbers from the confusion matrix, not just overall accuracy.
# A: 

# Q5: Go back to Exercise 1.1 Q5 — your prediction of which buildings
#     would be misclassified. Were you right? What does that tell you about
#     the value of drawing the data before modeling?
# A: 

---
## Part 6: The Ethics of Misclassification

Every classification error in this model has a human consequence.
This section asks you to think carefully about what those consequences are —
and who bears them.

This is not a coding exercise. It is the most important part of the notebook.

In [ ]:
# Scenario: Your smart city experiences a major hurricane.
# Grid capacity is reduced to 40% of normal for 72 hours.
# Your classification model is directing power allocation automatically.

# Calculate approximate power demand by tier
critical_mean   = df[df['priority_tier'] == 'Critical']['global_mean_kWh'].mean()
flexible_mean   = df[df['priority_tier'] == 'Flexible']['global_mean_kWh'].mean()
noncrit_mean    = df[df['priority_tier'] == 'Non_Critical']['global_mean_kWh'].mean()

n_critical   = len([b for b, t in priority_labels.items() if t == 'Critical'])
n_flexible   = len([b for b, t in priority_labels.items() if t == 'Flexible'])
n_noncrit    = len([b for b, t in priority_labels.items() if t == 'Non_Critical'])

total_critical = critical_mean * n_critical
total_flexible = flexible_mean * n_flexible
total_all      = df['global_mean_kWh'].mean() * 19

print("Hurricane Scenario — Power Budget Analysis")
print("=" * 50)
print(f"Total normal grid demand:     {total_all:.0f} kWh")
print(f"Available at 40% capacity:    {total_all * 0.4:.0f} kWh")
print(f"")
print(f"Critical tier demand ({n_critical} buildings):   {total_critical:.0f} kWh")
print(f"Critical as % of available:   {total_critical/(total_all*0.4)*100:.1f}%")
print(f"")
print(f"Critical + Flexible demand:   {total_critical + total_flexible:.0f} kWh")
print(f"As % of available:            {(total_critical+total_flexible)/(total_all*0.4)*100:.1f}%")

In [ ]:
# Exercise 6.1 — The hardest questions
# Answer all five. These belong in your Week 4 design proposal.

# Q1: Based on the power budget analysis above —
#     can your Critical tier be fully powered at 40% grid capacity?
#     If not, what does that mean for your classification system?
#     Do you need to add a fourth tier? Reduce Critical tier size?
# A: 

# Q2: Your model misclassified ___ buildings.
#     For EACH misclassified building, describe the real-world consequence
#     of that error during the 72-hour hurricane scenario.
#     Be specific — who is affected and how?
# A: 

# Q3: Your model learned priority tiers from labels YOUR TEAM assigned.
#     If a different team assigned different labels, they would get
#     a different model with different errors.
#     What does this tell you about the relationship between human decisions
#     and AI outputs? Who is responsible for the model's errors?
# A: 

# Q4: During the hurricane, a grocery store is classified as Non_Critical
#     and loses power. Emergency responders needed that store for food supplies.
#     Was this the model's fault? The labeler's fault? The system designer's fault?
#     What would you change?
# A: 

# Q5: Should an AI system be making these decisions automatically,
#     or should a human review every classification before it affects power delivery?
#     What is the tradeoff between speed and human oversight in an emergency?
# A: 

---
## Part 7: Connecting to Optimization

You now have two models:
- A **regression model** that predicts how much power each building needs
- A **classification model** that assigns each building a priority tier

In Week 4 you will combine these into an **optimization model** that decides
how to distribute finite power across your community — maximizing benefit
while respecting the priority order your classification defined.

The final reflection prepares you for that work.

In [ ]:
# Final Reflection — answer all four questions
# These answers belong in your Week 4 design proposal.

# Q1: Your classification model achieved ___% accuracy.
#     Your regression model achieved R² = ___.
#     Which model are you more confident in for emergency deployment?
#     Explain using specific metrics — not just which number is higher.
# A: 

# Q2: The decision tree asks questions about energy consumption patterns
#     to classify priority. But priority is fundamentally a human value judgment,
#     not a data pattern. 
#     What is the model actually learning — the right answer, or YOUR answer?
#     Is there a difference?
# A: 

# Q3: Name one type of building that your classification system cannot handle well.
#     (Think about buildings whose priority changes depending on context —
#     a school during a hurricane vs a normal day.)
#     How would you modify the system to handle context-dependent priority?
# A: 

# Q4: You have now drawn your data, built a regression model, and built
#     a classification model. You started with a blank canvas.
#     Looking at everything your team has built — what is the most important
#     thing you learned that you did NOT expect to learn?
# A: 

---
## Extension: Clustering — What If You Had No Labels?

*(Optional — required for teams competing for top marks)*

Your classification model learned from labels YOUR TEAM assigned.
But what if no one had decided on priority tiers ahead of time?

**Clustering** is an unsupervised technique — the algorithm finds natural
groupings in the data WITHOUT being told what the groups should be.
You then interpret what each cluster means.

Use K-Means clustering with k=3 on the same features you used for classification.
Compare the clusters the algorithm found to your human-assigned tiers.

**Key questions:**
- Did the algorithm find groups that match your priority tiers?
- Where did it disagree? What does that tell you?
- Which approach — human labels or data-driven clusters — would you trust more
  for a real emergency system? Why?

Write a 3-4 sentence analysis for your Week 4 design proposal.

In [ ]:
# Extension — Clustering
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# YOUR CODE HERE
# 1. Scale your features (KMeans is sensitive to scale)
#    scaler = StandardScaler()
#    X_scaled = scaler.fit_transform(X)
#
# 2. Fit KMeans with k=3
#    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
#    kmeans.fit(X_scaled)
#
# 3. Compare cluster assignments to your priority labels
#    Look at which building types land in each cluster
#
# 4. Visualize using a scatter plot of two features, colored by cluster

# YOUR ANALYSIS:
# 

---
## What Comes Next

You have now drawn the data, predicted demand with regression, and classified
users by priority. The three colors on your palette are mixed.

In Week 4 you will paint the finished picture: an optimization model that
distributes finite power across your smart city community — using everything
your team has built, debated, and learned.

Your regression predictions tell you how much each building needs.
Your classification tiers tell you who gets served first.
Your optimization model decides how to make those two things work together
when the grid cannot give everyone what they need.

---
*Quick Start Orange County: AI Design Contest — University of Florida*